# Pyrates FEP
# Notebook 07: LLM annotation

Annotation of the B/C feedback by frontier LLMs, with the same coding form as the experts
(`campaign/2_full/full_form.json`). Each model is treated as one more annotator: it receives the
same information about the student's activity as an expert on the FEP platform, plus the knowledge
of the game that the experts acquired by playing Pyrates and watching each level's solution video.
Few-shot examples come from the gold standard (expert agreement, completed by the adjudication of
notebook 06); the rest of the gold standard is held out to measure each model's agreement with the
experts.

Two layers of context, kept separate on purpose:
- **Trace layer, identical to FEP**: labels, details panel, feedback text cleaned as the platform
  does (`campaign/4_llm/platform_context.json`, exported from the webtool). Nothing the platform
  hides is sent (positions, group, student id, the generator's `<feedback_caractéristiques>` block).
- **Application layer, game knowledge**: game rules (`game_rules.json`), ASCII maps and level
  dynamics (`level_maps.json`), student-side interface and programming memo (`game_interface.json`).

### Inputs:
- `data/input/cleaned/cleaned_interaction_traces.csv` (notebook 01)
- `campaign/2_full/full_form.json`
- `campaign/4_llm/platform_context.json`, `game_rules.json`, `level_maps.json`, `game_interface.json`
- `campaign/2_full/full_result.csv` (expert codings, for the expert-expert agreement)
- `campaign/3_adjudication/gold.csv` (gold standard, notebook 06)
- API keys in a `.env` file at the project root (`ANTHROPIC_API_KEY`, `OPENAI_API_KEY`,
  `TYPESAFE_API_KEY`), never in the notebook

### Outputs:
- `campaign/4_llm/prompt_<version>.md`: rendered prompt and sample contexts, for review
- `campaign/4_llm/raw/<model>.jsonl`: raw API responses, used as a cache
- `campaign/4_llm/llm_result.csv`: annotations in the FEP export format, one row per (trace, model)
- `campaign/4_llm/few_shot_examples.csv`: trace ids used as prompt examples
- `campaign/4_llm/dev_scores.csv`: dev-set agreement of each prompt version, one row per (version, annotator, question)

### Quick reference: what to tune

| Parameter | Default | What it does |
|---|---|---|
| `DRY_RUN` | `True` | No API call at all: contexts, prompt and cost estimates only. Set to `False` to call the APIs. |
| `PROMPT_VERSION` | `"v1"` | Part of the cache key: bump it whenever the prompt changes, so old responses are not reused. |
| `MODELS` | 3 models | Model id to provider. Remove an entry to skip a model. |
| `ANTHROPIC_EFFORT`, `OPENAI_REASONING_EFFORT` | `"high"` | Effort level for Claude and GPT Sol (both default to `medium`), aligned for a fair comparison. |
| `SPLIT_SEED`, `SPLIT_GAMES` | `2026`, 5 / 8 | Gold games drawn for the example pool and the dev set; the other gold games form the test set. |
| `N_EXAMPLES` | `10` | Number of few-shot examples selected from the example pool. |
| `FEW_SHOT_TRACE_IDS` | `[]` | Explicit list of example trace ids; overrides the automatic selection when not empty. |
| `JEV_THRESHOLD` | `0.5` | Probability above which a JEV yes/no answer counts as a ticked category. |
| `JEV_N_EXAMPLES` | `3` | Number of examples given to Jev (the shortest ones), to stay under its 32k-token limit. |
| `RUN_SPLITS` | `["dev"]` | Which subsets the annotation cell processes. |
| `EVAL_SPLIT` | `"dev"` | Subset evaluated in section 9: `"dev"` while tuning the prompt, `"test"` once, with the frozen prompt. |

## 1. Setup and imports

In [ ]:
from __future__ import annotations

import hashlib
import json
import random
import re
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import cohen_kappa_score

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Paths relative to the notebook: notebooks/ -> project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
INPUT_PATH = DATA_DIR / "input" / "cleaned" / "cleaned_interaction_traces.csv"
CAMPAIGN_DIR = PROJECT_ROOT / "campaign"
FORM_PATH = CAMPAIGN_DIR / "2_full" / "full_form.json"
EXPERT_RESULT_PATH = CAMPAIGN_DIR / "2_full" / "full_result.csv"
GOLD_PATH = CAMPAIGN_DIR / "3_adjudication" / "gold.csv"
LLM_DIR = CAMPAIGN_DIR / "4_llm"
PLATFORM_PATH = LLM_DIR / "platform_context.json"
GAME_RULES_PATH = LLM_DIR / "game_rules.json"
LEVEL_MAPS_PATH = LLM_DIR / "level_maps.json"
GAME_INTERFACE_PATH = LLM_DIR / "game_interface.json"
RAW_DIR = LLM_DIR / "raw"
OUTPUT_PATH = LLM_DIR / "llm_result.csv"
FEW_SHOT_PATH = LLM_DIR / "few_shot_examples.csv"
DEV_SCORES_PATH = LLM_DIR / "dev_scores.csv"

load_dotenv(PROJECT_ROOT / ".env")

print("Project root :", PROJECT_ROOT.name)
for path in [INPUT_PATH, FORM_PATH, PLATFORM_PATH, GAME_RULES_PATH, LEVEL_MAPS_PATH, GAME_INTERFACE_PATH, EXPERT_RESULT_PATH, GOLD_PATH]:
    print(f"{str(path.relative_to(PROJECT_ROOT)):45s} exists: {path.exists()}")

In [ ]:
DRY_RUN = True                 # True: no API call, only contexts, prompt and estimates
PROMPT_VERSION = "v1"          # part of the cache key: bump it whenever the prompt changes

MODELS = {                     # model id -> provider
    "claude-opus-5-5": "anthropic",
    "gpt-6.1-sol": "openai",
    "jev-1.13.0": "typesafe",
}
ANTHROPIC_EFFORT = "high"      # Claude Opus 5.5 defaults to "medium"
OPENAI_REASONING_EFFORT = "high"   # GPT Sol defaults to "medium"; aligned with Claude
MAX_OUTPUT_TOKENS = 16000

SPLIT_SEED = 2026
SPLIT_GAMES = {"examples": 5, "dev": 8}   # gold games per subset; the remaining gold games are the test set
N_EXAMPLES = 10
FEW_SHOT_TRACE_IDS: list[str] = []       # explicit example trace ids (overrides the automatic selection)

JEV_THRESHOLD = 0.5
JEV_MAX_STATE_TOKENS = 32000             # Jev 1.13 limit for the state plus the longest question
JEV_N_EXAMPLES = 3                       # Jev gets the shortest examples only, to stay under its limit

RUN_SPLITS = ["dev"]                     # subsets processed by the annotation cell (section 7)
EVAL_SPLIT = "dev"                       # "dev" while tuning the prompt, "test" once for the final scores

## 2. Inputs

In [ ]:
traces = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False, low_memory=False)
traces["_id"] = traces["id"].astype(int)
traces["_level"] = traces["level_id"].astype(int)
# FEP order: date (millisecond precision), then source trace id
traces = traces.sort_values(["game_id", "date", "_id"]).reset_index(drop=True)

form = json.load(open(FORM_PATH, encoding="utf-8"))
platform = json.load(open(PLATFORM_PATH, encoding="utf-8"))
game_rules = json.load(open(GAME_RULES_PATH, encoding="utf-8"))
level_maps = json.load(open(LEVEL_MAPS_PATH, encoding="utf-8"))
game_interface = json.load(open(GAME_INTERFACE_PATH, encoding="utf-8"))

expert_result = pd.read_csv(EXPERT_RESULT_PATH, dtype=str, keep_default_na=False) if EXPERT_RESULT_PATH.exists() else None
gold = pd.read_csv(GOLD_PATH, dtype=str, keep_default_na=False) if GOLD_PATH.exists() else None

print(f"Traces: {len(traces)} in {traces.game_id.nunique()} games")
print(f"Coding form: {form['name']} ({len(form['questions'])} questions)")
print(f"FEP context: {len(platform['levels'])} level sheets, {len(platform['memo'])} memo sections")
print(f"Expert codings: {'%d rows' % len(expert_result) if expert_result is not None else 'not available yet'}")
print(f"Gold standard: {'%d feedback' % len(gold) if gold is not None else 'not available yet'}")

## 3. Annotation units and gold split

One unit per feedback actually delivered to a student: a `RECEIVED` trace on `ASSISTANT_HELP`
in group `B` or `C`. The group is used here only to stratify the split; it never reaches the
models.

The gold standard plays two roles, kept apart **by game**: an example pool (few-shot examples),
a dev set (prompt tuning) and a test set (final evaluation, run once the prompt is frozen). A game
never contributes to two subsets, so a model is never evaluated on a game it saw an example from.
Units outside the gold standard form the production set. Feedback removed from the gold standard
(arbiters in disagreement) stay in their game's subset but have no reference label.

In [ ]:
is_feedback = (traces.action_name == "RECEIVED") & (traces.object_name == "ASSISTANT_HELP")
units = traces.loc[is_feedback & traces.group_id.isin(["B", "C"]),
                   ["id", "game_id", "level_id", "group_id"]].rename(columns={"id": "trace_id"})
units = units.reset_index(drop=True)

split_of_game: dict[str, str] = {}
if expert_result is not None:
    gold_games = sorted(set(expert_result.game_id))   # every game coded by the experts
    group_of_game = units.drop_duplicates("game_id").set_index("game_id").group_id
    rng = random.Random(SPLIT_SEED)
    by_group = {g: [x for x in gold_games if group_of_game.get(x) == g] for g in ["B", "C"]}
    for games in by_group.values():
        rng.shuffle(games)
    # interleave B and C so that each subset stays balanced between groups
    interleaved = [g for pair in zip(by_group["B"], by_group["C"]) for g in pair]
    longest = max(by_group.values(), key=len)
    interleaved += longest[min(len(by_group["B"]), len(by_group["C"])):]
    cursor = 0
    for split, n_games in SPLIT_GAMES.items():
        for game_id in interleaved[cursor:cursor + n_games]:
            split_of_game[game_id] = split
        cursor += n_games
    for game_id in interleaved[cursor:]:
        split_of_game[game_id] = "test"

units["split"] = units.game_id.map(split_of_game).fillna("production")

summary_split = units.groupby(["split", "group_id"]).agg(games=("game_id", "nunique"), feedback=("trace_id", "size"))
print(summary_split.to_string())
assert units.groupby("game_id").split.nunique().max() == 1, "a game spans two subsets"

## 4. Context rendering

The functions below rebuild, as text, what an expert sees in the FEP workspace for one feedback:

1. **Partie**: game id, current level, feedback rank `n/N` within the whole game.
2. **Traces**: the window set by question 1 of the form, from the previous feedback in the same
   level (or the start of the level for the first one) up to the help request. One line per
   trace in the FEP list format, followed by the content of the FEP "Détails trace" panel.
3. **Demande**: the current code at the help request (`ASKED` trace paired with the feedback).
4. **Contenu feedback**: the feedback text cleaned exactly as the FEP platform displays it.

Labels, display rules, ASKED/RECEIVED pairing and the feedback counter follow
`platform_context.json`. Game knowledge (level sheets, maps, rules, memo) goes in the static
part of the prompt (section 5).

In [ ]:
ACTION_LABELS = platform["action_labels"]
OBJECT_LABELS = platform["object_labels"]
REASON_LABELS = platform["reason_labels"]
TEXTS = platform["detail_texts"]
FEP_MEMO = {m["object_name"]: m for m in platform["memo"]}

LINE_ERROR_TYPES = {"GAME_ERROR_PROGRAM", "SYNTACTIC_ERROR_PROGRAM", "SEMANTIC_ERROR_PROGRAM"}
NO_RUN_PROGRESSION = {"SYNTACTIC_ERROR_PROGRAM", "TOO_MANY_LINES_ERROR_PROGRAM"}


def unescape(text: str) -> str:
    return text.replace("\\n", "\n").replace("\\t", "\t")


def object_label(row) -> str:
    if row.object_name == "LEVEL":
        return OBJECT_LABELS["LEVEL"].replace("{n}", row.level_id)
    if row.object_name in FEP_MEMO:
        memo = FEP_MEMO[row.object_name]
        return OBJECT_LABELS["MEMO_SECTION"].replace("{parent}", memo["parent"]).replace("{title}", memo["title"])
    return OBJECT_LABELS.get(row.object_name, row.object_name)


def numbered_code(code_text: str, highlighted: set[int]) -> str:
    lines = unescape(code_text).split("\n")
    return "\n".join(
        f"{'>>' if i in highlighted else '  '} {i:>2} | {line}" for i, line in enumerate(lines, start=1)
    )


def last_nonempty_lines(code_text: str, count: int) -> set[int]:
    lines = unescape(code_text).split("\n")
    nonempty = [i for i, line in enumerate(lines, start=1) if line.strip()]
    return set(nonempty[-count:]) if count > 0 else set()


def mmss(seconds: str) -> str:
    total = int(float(seconds or 0))
    return f"{total // 60:02d}:{total % 60:02d}"


def render_trace(row, asked_code: dict[str, str]) -> str:
    """One FEP list row, then the content of the FEP 'Détails trace' panel."""
    head = (f"[{row.date[2:19]} | N{row.level_id} | {mmss(row.game_time)}] "
            f"{ACTION_LABELS.get(row.action_name, row.action_name)} | {object_label(row)}")
    details = []
    progression = TEXTS["progression"]
    if row.action_name == "LAUNCHED" and row.object_name not in NO_RUN_PROGRESSION:
        progression = TEXTS["progression_before_run"]
    details.append(f"{progression}{row.progression or 0}%")
    if row.error_message:
        details.append(f"{TEXTS['error_message']} : {unescape(row.error_message)}")
    if row.action_name == "CHANGED" and row.object_name == "EXECUTION_SPEED_SLIDE" and row.value:
        details.append(f"{TEXTS['speed']} : x{int(row.value) / 10:.1f}")
    if row.action_name == "DISPLAYED" and row.duration:
        details.append(f"{TEXTS['display_time']} : {row.duration}s (contenu de la section : voir le mémo)")
    if row.object_name == "LEVEL_LOST_PROGRAM" and row.reason_name:
        details.append(f"{TEXTS['reason']} : {REASON_LABELS.get(row.reason_name, row.reason_name)}")
    if row.action_name == "LAUNCHED":
        highlighted: set[int] = set()
        if row.object_name in LINE_ERROR_TYPES and row.value.isdigit():
            highlighted = {int(row.value)}
        if row.object_name == "TOO_MANY_LINES_ERROR_PROGRAM" and row.value.isdigit():
            details.append(f"{TEXTS['too_many_lines']} : {row.value}")
            highlighted = last_nonempty_lines(row.code, int(row.value))
        details.append(f"{TEXTS['executed_code']} :\n" + (numbered_code(row.code, highlighted) if row.code.strip() else TEXTS["empty_editor"]))
    elif row.action_name == "ASKED":
        details.append(f"{TEXTS['current_code']} :\n" + (numbered_code(row.code, set()) if row.code.strip() else TEXTS["empty_editor"]))
    elif row.action_name == "RECEIVED" and row.object_name == "ASSISTANT_HELP":
        current = asked_code.get(row.id)
        if current is not None:
            details.append(f"{TEXTS['current_code']} :\n" + (numbered_code(current, set()) if current.strip() else TEXTS["empty_editor"]))
    elif row.action_name in ("COPIED", "PASTED") and row.code:
        label = TEXTS["copied_code"] if row.action_name == "COPIED" else TEXTS["pasted_code"]
        details.append(f"{label} :\n" + unescape(row.code))
    body = "\n".join("    " + line for block in details for line in block.split("\n"))
    return f"{head}\n{body}"

In [ ]:
def pair_asked_received(game: pd.DataFrame) -> dict[str, str]:
    """RECEIVED id -> id of its ASKED, with the importer's rule (same level first, each ASKED used once)."""
    pending: list[tuple[str, str]] = []   # (asked id, level)
    pairs: dict[str, str] = {}
    for row in game.itertuples():
        if row.object_name != "ASSISTANT_HELP":
            continue
        if row.action_name == "ASKED":
            pending.append((row.id, row.level_id))
        elif row.action_name == "RECEIVED" and pending:
            same_level = [i for i, (_, lvl) in enumerate(pending) if lvl == row.level_id]
            index = same_level[-1] if same_level else len(pending) - 1
            pairs[row.id] = pending.pop(index)[0]
    return pairs


def clean_feedback(text: str) -> str:
    """Same transformations as the FEP feedback display (markup tags in_line/block/memo kept)."""
    cleaned = unescape(text)
    cleaned = re.sub(r"<feedback_caract[ée]ristiques>.*?</feedback_caract[ée]ristiques>", "", cleaned, flags=re.S)
    cleaned = re.sub(r"</?feedback(?:_message)?>", "", cleaned)
    cleaned = cleaned.lstrip("﻿​‎")
    parts = re.split(r"(<in_line>.*?</in_line>|<block>.*?</block>)", cleaned, flags=re.S)
    cleaned = "".join(p if p.startswith(("<in_line>", "<block>")) else re.sub(r"\?{3,}", "🙂", p) for p in parts)
    return cleaned.strip()


games = {game_id: rows.reset_index(drop=True) for game_id, rows in traces.groupby("game_id", sort=False)}
asked_of = {}
for game in games.values():
    asked_of.update(pair_asked_received(game))
code_of = dict(zip(traces.id, traces.code))
asked_code = {received: code_of[asked] for received, asked in asked_of.items()}


def window_rows(trace_id: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """(window traces up to the help request, game) for one feedback."""
    game_id = traces.loc[traces.id == trace_id, "game_id"].iat[0]
    game = games[game_id]
    position = game.index[game.id == trace_id][0]
    level = game.at[position, "level_id"]
    before = game.iloc[:position]
    previous_feedback = before[(before.action_name == "RECEIVED") & (before.object_name == "ASSISTANT_HELP")
                               & (before.level_id == level)]
    if len(previous_feedback):
        start = previous_feedback.index[-1] + 1
    else:
        level_start = before[(before.object_name == "LEVEL") & (before.action_name == "STARTED") & (before.level_id == level)]
        start = level_start.index[-1] if len(level_start) else before.index[before.level_id == level].min()
        start = 0 if pd.isna(start) else start
    asked_id = asked_of.get(trace_id)
    end = game.index[game.id == asked_id][0] if asked_id else position - 1
    return game.iloc[start:end + 1], game


def render_feedback_context(trace_id: str) -> str:
    window, game = window_rows(trace_id)
    row = game[game.id == trace_id].iloc[0]
    feedback_ids = game[(game.action_name == "RECEIVED") & (game.object_name == "ASSISTANT_HELP")].id.tolist()
    rank = feedback_ids.index(trace_id) + 1
    trace_text = "\n\n".join(render_trace(r, asked_code) for r in window.itertuples() if r.id != asked_of.get(trace_id))
    current = asked_code.get(trace_id)
    request = ("Demandé | Aide de l'assistant\n    " + TEXTS["current_code"] + " :\n"
               + "\n".join("    " + line for line in (numbered_code(current, set()) if current and current.strip() else TEXTS["empty_editor"]).split("\n"))
               if current is not None else "(aucune demande associée)")
    return (
        f"<partie>\nPartie {row.game_id} | Niveau {row.level_id} | Contenu feedback ({rank}/{len(feedback_ids)})\n</partie>\n\n"
        f"<traces>\n{trace_text or '(aucune trace entre le début de la fenêtre et la demande)'}\n</traces>\n\n"
        f"<demande>\n{request}\n</demande>\n\n"
        f"<feedback>\n{clean_feedback(row.code)}\n</feedback>"
    )


# Checks: nothing hidden by FEP leaks into the rendered contexts
sample_ids = units.groupby("level_id").trace_id.first().tolist()
for tid in sample_ids:
    ctx = render_feedback_context(tid)
    assert "feedback_caract" not in ctx and "<feedback_message>" not in ctx
    assert "x_pos" not in ctx and "student_id" not in ctx
print(f"OK: {len(sample_ids)} sample contexts rendered (one per level), no hidden field.")
print(f"Feedback with a paired help request: {units.trace_id.isin(asked_code).mean():.1%}")

## 5. Prompt and output schema

The prompt has a **static part**, identical for every call, and a **per-feedback part**
(section 4). The static part carries the role, the coding form, the game rules, the game
interface and memo, the 8 level sheets with their maps, then the few-shot examples. Claude and
GPT cache it, so its cost is paid almost once.

The output schema is generated from `full_form.json`: a short rationale, one value for the stage,
lists of values for the multiple-choice questions, a free text per "other" option. Exclusivity
(`R0-NONE`, `I0-NONE`) is checked after the call.

In [ ]:
QUESTIONS = form["questions"]


def form_text() -> str:
    blocks = []
    for n, q in enumerate(QUESTIONS, start=1):
        w = q["widget"]
        kind = "Choix unique" if w["type"] == "RADIO" else "Choix multiple"
        hint = re.sub(r"^(Choix unique|Choix multiple)\.\s*", "", w["label"]["fr"])
        lines = [f"Q{n}. [{q['tag']['fr']}] {q['label']['fr']} (champ `{q['id']}`, {kind.lower()})", f"Consigne : {hint}"]
        for o in w["options"]:
            flags = []
            if o.get("exclusive"):
                flags.append("exclusif : ne peut pas être coché avec une autre option")
            if o.get("allowsFreeText"):
                flags.append(f"préciser dans `{q['id']}_other`")
            suffix = f" ({'; '.join(flags)})" if flags else ""
            lines.append(f"- {o['value']} : {o['label']['fr']}{suffix}. {o['description']['fr']}")
        blocks.append("\n".join(lines))
    return "\n\n".join(blocks)


def game_rules_text() -> str:
    parts = ["Exécution d'un programme :"] + [f"- {r}" for r in game_rules["execution"]]
    parts.append("\nFonctions de contrôle (niveaux où elles sont disponibles) :")
    parts += [f"- {f['function']} (niveaux {', '.join(map(str, f['levels']))}) : {f['effect']}" for f in game_rules["functions"]]
    limits = ", ".join(f"niveau {k} : {v if v else 'aucune'}" for k, v in game_rules["line_limits"]["by_level"].items())
    parts.append(f"\nNombre maximal de lignes du programme : {limits}.")
    return "\n".join(parts)


def interface_text() -> str:
    ui = game_interface["interface"]
    assistant = ui["assistant_panel"]
    console = ui["console_messages"]
    return "\n".join([
        "Éléments de l'écran de jeu : " + "; ".join(ui["buttons_and_panels"]) + ".",
        f"Panneau de l'assistant numérique « {assistant['name']} » : bouton « {assistant['button']} », "
        f"libellé « {assistant['unavailable_label']} » après une demande. Texte affiché : {assistant['introduction']}",
        "Messages de la console : " + "; ".join([console["execution_launched"], console["execution_completed"], console["execution_stopped"]]) + ".",
        "Messages d'erreur du jeu : " + "; ".join(console["game_errors"] + [console["too_many_lines"]]) + ".",
        "Messages d'erreur de syntaxe propres au jeu : " + "; ".join(console["syntax_messages"]),
    ])


def memo_text() -> str:
    memo = game_interface["programming_memo"]
    blocks = []
    for cat in memo["categories"]:
        lines = [f"### {cat['category']} ({cat['summary']})", f"Utilité : {cat['utility']}"]
        for s in cat["sections"]:
            lines.append(f"\n#### {cat['category']} > {s['title']}  [{s['object_name']}]")
            if s.get("description"):
                lines.append(s["description"])
            if s.get("model"):
                lines.append(f"Modèle Python :\n{s['model']['python']}")
                if s["model"].get("scratch"):
                    lines.append(f"Modèle Scratch :\n{s['model']['scratch']}")
            for e in s.get("examples", []):
                if e.get("python"):
                    lines.append(f"Exemple Python :\n{e['python']}")
                if e.get("scratch"):
                    lines.append(f"Exemple Scratch :\n{e['scratch']}")
            lines += [f"- {r}" for r in s.get("remarks", [])]
        blocks.append("\n".join(lines))
    return "\n\n".join(blocks)


def map_text(level_map: dict) -> str:
    rows = level_map["ascii"]
    width = len(rows[0])
    tens = "".join(str(c // 10) if c >= 10 else " " for c in range(width))
    units_row = "".join(str(c % 10) for c in range(width))
    grid = [f"     {tens}", f"     {units_row}"] + [f"  {r:>2} {line}" for r, line in enumerate(rows)]
    return "\n".join(grid)


def level_sheet(level_id: int) -> str:
    fep = next(l for l in platform["levels"] if l["level_id"] == level_id)
    student = next(l for l in game_interface["level_panels"] if l["level_id"] == level_id)
    lmap = next(l for l in level_maps["levels"] if l["level_id"] == level_id)
    labels = platform["game_panel_labels"]
    lines = [f'<niveau id="{level_id}">', "Fiche expert (panneau Jeu de FEP) :",
             f"- {labels['objective']} : {fep['objective']}",
             f"- {labels['control_functions']} {', '.join(fep['control_functions'])}",
             f"- {labels['target_concept']} {fep['target_concept']}",
             f"- {labels['secondary_concept']} {', '.join(fep['secondary_concepts']) or 'aucun'}",
             f"- {labels['limiting_constraint']} {fep['limiting_constraint']}",
             f"- {labels['constraint_indicators']}"]
    lines += [f"    - {i}" for i in fep["constraint_indicators"]]
    for label, value in fep.get("other_panel_fields", {}).items():
        lines.append(f"- {label} :\n{value}")
    lines.append(f"- {labels['expected_solution']} :\n{fep['expected_solution']}")
    lines.append("\nPanneau Niveaux vu par l'élève dans le jeu :")
    lines.append(f"- Objectif : {student['objective']}")
    if student["constraints"]:
        lines.append(f"- Contraintes : {student['constraints']}")
    lines.append("- Fonctions de contrôle :")
    lines += [f"    - {f['usage']} : {f['description']}" + (f" Exemple : {f['example']}" if f.get("example") else "") for f in student["control_functions"]]
    lines.append(f"\nCarte (une case par bloc ; ligne 0 en haut, colonne 0 à gauche ; légende commune) :\n{map_text(lmap)}")
    lines.append(f"Sens initial du pirate : tourné vers la {lmap['pirate_facing']}.")
    if lmap.get("random"):
        lines.append(f"Change à chaque exécution : {lmap['random']}")
    if lmap.get("notes"):
        lines.append(f"Dynamique du niveau : {lmap['notes']}")
    lines.append("</niveau>")
    return "\n".join(lines)

In [ ]:
FEEDBACK_MARKUP_HELP = (
    "Balises du feedback : `<in_line>...</in_line>` est affiché comme du code dans le texte ; "
    "`<block>...</block>` comme un bloc de code Python ; `<memo>...</memo>` comme un badge bleu portant le "
    "nom d'une section du mémo écrit par l'assistant (badge non cliquable : le nom peut ne correspondre à "
    "aucune section existante). Le reste du feedback est du Markdown."
)

SYSTEM_PROMPT = f"""<role>
Tu es un expert de la campagne d'annotation FEP. Tu codes les feedbacks produits par l'assistant numérique
« Coco » du jeu sérieux Pyrates, qui apprend la programmation Python à des élèves de collège et de lycée.
Tu as joué à Pyrates et vu la vidéo de solution de chaque niveau. Pour chaque feedback, tu reçois exactement
ce qu'un expert voit sur la plateforme FEP : la partie, les traces de l'élève, sa demande d'aide et le
contenu du feedback. Tu réponds aux quatre questions de la grille de codage, en suivant leurs consignes à la
lettre, comme le ferait un expert humain.
</role>

<presentation_du_contexte>
Chaque message contient quatre blocs :
- `<partie>` : identifiant de partie, niveau courant, rang du feedback dans la partie.
- `<traces>` : l'activité de l'élève depuis le feedback précédent dans le même niveau, ou depuis le début du
  niveau s'il s'agit du premier feedback, jusqu'à la demande d'aide. Une trace par paragraphe : la ligne
  [date | niveau | temps de jeu] Action | Objet, puis le détail affiché par FEP. Dans le code, la ligne
  marquée `>>` est la ligne surlignée par FEP (ligne de l'erreur, ou lignes excédentaires). Les consultations
  du mémo indiquent la section et le temps d'affichage ; le contenu des sections est dans `<memo_du_jeu>`.
- `<demande>` : le code de l'élève au moment où il a demandé de l'aide.
- `<feedback>` : le feedback à coder. {FEEDBACK_MARKUP_HELP}
</presentation_du_contexte>

<grille_de_codage>
{form_text()}
</grille_de_codage>

<regles_du_jeu>
{game_rules_text()}
</regles_du_jeu>

<interface_du_jeu>
{interface_text()}
</interface_du_jeu>

<legende_des_cartes>
{json.dumps(level_maps['legend'], ensure_ascii=False, indent=0)}
</legende_des_cartes>

<niveaux>
{chr(10).join(level_sheet(i) for i in range(1, 9))}
</niveaux>

<memo_du_jeu>
{memo_text()}
</memo_du_jeu>

<format_de_reponse>
Réponds uniquement avec l'objet JSON demandé. Le champ `rationale` justifie brièvement tes choix (quelques
phrases), avant les codes. Utilise exactement les codes de la grille. Ne remplis un champ `_other` que si
l'option « Autre » correspondante est cochée ; sinon laisse-le vide.
</format_de_reponse>"""


def build_output_schema() -> dict:
    properties = {"rationale": {"type": "string"}}
    for q in QUESTIONS:
        values = [o["value"] for o in q["widget"]["options"]]
        if q["widget"]["type"] == "RADIO":
            properties[q["id"]] = {"type": "string", "enum": values}
        else:
            properties[q["id"]] = {"type": "array", "items": {"type": "string", "enum": values}}
        if any(o.get("allowsFreeText") for o in q["widget"]["options"]):
            properties[f"{q['id']}_other"] = {"type": "string"}
    return {"type": "object", "properties": properties, "required": list(properties), "additionalProperties": False}


OUTPUT_SCHEMA = build_output_schema()
FEP_COLUMNS = ["game_id", "trace_id", "expert_id"] + [c for c in OUTPUT_SCHEMA["properties"] if c != "rationale"]
print("Output columns:", FEP_COLUMNS)
print(f"Static prompt: {len(SYSTEM_PROMPT):,} characters (~{len(SYSTEM_PROMPT) // 3.5:,.0f} tokens)")

### Few-shot examples

Examples are drawn from the example pool only, with their gold labels (expert agreement or
adjudication). The greedy selection takes, at each step, the feedback that adds the most codes of
the form not covered yet; ties go to the group (B or C), then the level, least represented so far.
It stops at `N_EXAMPLES`. The cell reports the codes left without any example (rare codes may be
absent from the pool) and the spread of the examples across groups and levels. The list stays empty
while the gold standard is not available.

In [ ]:
def codes_of(rec) -> set[str]:
    return {v for q in QUESTIONS for v in str(rec[q["id"]]).split(",") if v}


def select_examples(pool: pd.DataFrame, n: int) -> list[str]:
    """Greedy: each step takes the feedback adding the most uncovered codes; ties go to the group,
    then the level, least represented so far, then to the shuffled order."""
    level_of = dict(zip(units.trace_id, units.level_id))
    group_of = dict(zip(units.trace_id, units.group_id))
    chosen, covered = [], set()
    groups, levels = Counter(), Counter()
    candidates = pool.sample(frac=1, random_state=SPLIT_SEED).to_dict("records")
    while candidates and len(chosen) < n:
        best = max(candidates, key=lambda r: (len(codes_of(r) - covered),
                                              -groups[group_of[r["trace_id"]]],
                                              -levels[level_of[r["trace_id"]]]))
        candidates.remove(best)
        chosen.append(best["trace_id"])
        covered |= codes_of(best)
        groups[group_of[best["trace_id"]]] += 1
        levels[level_of[best["trace_id"]]] += 1
    return chosen


examples: list[dict] = []
if gold is not None:
    reference = gold
    pool = reference[reference.trace_id.isin(units.loc[units.split == "examples", "trace_id"])]
    example_ids = FEW_SHOT_TRACE_IDS or select_examples(pool, N_EXAMPLES)
    for tid in example_ids:
        rec = reference[reference.trace_id == tid].iloc[0]
        answer = {"rationale": ""}
        for q in QUESTIONS:
            value = rec[q["id"]]
            answer[q["id"]] = value if q["widget"]["type"] == "RADIO" else [v for v in value.split(",") if v]
            if f"{q['id']}_other" in OUTPUT_SCHEMA["properties"]:
                answer[f"{q['id']}_other"] = rec.get(f"{q['id']}_other", "")
        examples.append({"trace_id": tid, "context": render_feedback_context(tid), "answer": answer})
    pd.DataFrame({"trace_id": [e["trace_id"] for e in examples]}).to_csv(FEW_SHOT_PATH, index=False)

print(f"Few-shot examples: {len(examples)}")
if examples:
    chosen_rows = gold.set_index("trace_id").loc[[e["trace_id"] for e in examples]].reset_index()
    all_codes = [o["value"] for q in QUESTIONS for o in q["widget"]["options"]]
    covered = set().union(*(codes_of(r) for r in chosen_rows.to_dict("records")))
    missing = [c for c in all_codes if c not in covered]
    print(f"Codes covered: {len(covered)} / {len(all_codes)}")
    print(f"Codes without any example: {', '.join(missing) or 'none'}")
    chosen_units = units.set_index("trace_id").loc[chosen_rows.trace_id]
    print("Examples per group:", dict(Counter(chosen_units.group_id)))
    print("Examples per level:", dict(sorted(Counter(chosen_units.level_id.astype(int)).items())))

In [ ]:
def validate_answer(answer: dict) -> tuple[dict, list[str]]:
    """Normalizes a model answer and lists the problems found (exclusivity, empty choice, stray free text)."""
    issues, clean = [], {}
    for q in QUESTIONS:
        options = q["widget"]["options"]
        allowed = [o["value"] for o in options]
        if q["widget"]["type"] == "RADIO":
            value = answer.get(q["id"])
            if value not in allowed:
                issues.append(f"{q['id']}: invalid value {value!r}")
                value = ""
            clean[q["id"]] = value
        else:
            values = [v for v in allowed if v in set(answer.get(q["id"]) or [])]
            exclusive = [o["value"] for o in options if o.get("exclusive")]
            if exclusive and exclusive[0] in values and len(values) > 1:
                issues.append(f"{q['id']}: exclusive option with others, exclusive option dropped")
                values = [v for v in values if v != exclusive[0]]
            if not values:
                issues.append(f"{q['id']}: no option ticked")
            clean[q["id"]] = ",".join(values)
        other_field = f"{q['id']}_other"
        if other_field in OUTPUT_SCHEMA["properties"]:
            other_value = next(o["value"] for o in options if o.get("allowsFreeText"))
            text = (answer.get(other_field) or "").strip()
            clean[other_field] = text if other_value in clean[q["id"]].split(",") else ""
    return clean, issues

### Prompt preview

Rendered static prompt and three sample contexts (different levels, outside the gold standard),
written to `campaign/4_llm/prompt_<version>.md` for review against the FEP screen.

In [ ]:
preview_units = (units[units.split == "production"]
                 .assign(_level=lambda d: d.level_id.astype(int))
                 .query("_level in [1, 4, 8]")
                 .groupby("_level").nth(2))
preview_ids = preview_units.trace_id.tolist()

preview_path = LLM_DIR / f"prompt_{PROMPT_VERSION}.md"
with open(preview_path, "w", encoding="utf-8") as f:
    f.write(f"# Prompt {PROMPT_VERSION}\n\n## Static part (system prompt)\n\n````text\n{SYSTEM_PROMPT}\n````\n")
    f.write(f"\n## Few-shot examples\n\n{len(examples)} example(s).\n")
    for tid in preview_ids:
        f.write(f"\n## Sample context: trace {tid}\n\n````text\n{render_feedback_context(tid)}\n````\n")
print(f"Written: {preview_path.relative_to(PROJECT_ROOT)} (sample traces: {', '.join(preview_ids)})")

## 6. Model adapters

- **Claude** (`anthropic`): Messages API, JSON schema output (`output_config.format`), adaptive
  thinking with an explicit effort, static prompt cached for one hour. No server-side fallback to
  another model: a refusal is kept as a missing annotation, so every row is produced by the model
  it is attributed to.
- **GPT** (`openai`): Responses API with a strict JSON schema and the same effort level as Claude;
  OpenAI caches long prompt prefixes automatically.
- **Jev** (`typesafe`): no text generation. Question 1 is a `Choice` over the six stages; each
  category of questions 2 to 4 is a `Noul` (probability that the category applies), all in one call.
  The state is a JSON object with named fields, restricted to the current level to stay under the
  32k-token limit. The questions use the English wording of the form, Jev's primary language.
  A category counts as ticked above `JEV_THRESHOLD`; `R0-NONE` / `I0-NONE` are inferred when no other
  category is ticked; "other" categories get no free text.

In [ ]:
_clients: dict[str, object] = {}


def client_for(provider: str):
    if provider not in _clients:
        if provider == "anthropic":
            import anthropic
            _clients[provider] = anthropic.Anthropic()
        elif provider == "openai":
            import openai
            _clients[provider] = openai.OpenAI()
        elif provider == "typesafe":
            from typesafe_sdk import TypeSafeClient
            _clients[provider] = TypeSafeClient()
    return _clients[provider]


def anthropic_params(model: str, trace_id: str) -> dict:
    messages = []
    for ex in examples:
        messages += [{"role": "user", "content": ex["context"]},
                     {"role": "assistant", "content": json.dumps(ex["answer"], ensure_ascii=False)}]
    messages.append({"role": "user", "content": render_feedback_context(trace_id)})
    return {
        "model": model,
        "max_tokens": MAX_OUTPUT_TOKENS,
        "system": [{"type": "text", "text": SYSTEM_PROMPT, "cache_control": {"type": "ephemeral", "ttl": "1h"}}],
        "messages": messages,
        "output_config": {"effort": ANTHROPIC_EFFORT, "format": {"type": "json_schema", "schema": OUTPUT_SCHEMA}},
    }


def anthropic_parse(message: dict) -> dict | None:
    if message.get("stop_reason") != "end_turn":
        return None
    text = next((b["text"] for b in message["content"] if b["type"] == "text"), None)
    return json.loads(text) if text else None


def openai_params(model: str, trace_id: str) -> dict:
    inputs = []
    for ex in examples:
        inputs += [{"role": "user", "content": ex["context"]},
                   {"role": "assistant", "content": json.dumps(ex["answer"], ensure_ascii=False)}]
    inputs.append({"role": "user", "content": render_feedback_context(trace_id)})
    return {
        "model": model,
        "instructions": SYSTEM_PROMPT,
        "input": inputs,
        "reasoning": {"effort": OPENAI_REASONING_EFFORT},
        "text": {"format": {"type": "json_schema", "name": "fep_annotation", "schema": OUTPUT_SCHEMA, "strict": True}},
    }


def openai_parse(response: dict) -> dict | None:
    texts = [c["text"] for item in response.get("output", []) if item.get("type") == "message"
             for c in item.get("content", []) if c.get("type") == "output_text"]
    return json.loads(texts[-1]) if texts else None

In [ ]:
def jev_state(trace_id: str) -> dict:
    level_id = int(traces.loc[traces.id == trace_id, "level_id"].iat[0])
    level_functions = [f for f in game_rules["functions"] if level_id in f["levels"]]
    memo_titles = [f"{c['category']} > {s['title']}" for c in game_interface["programming_memo"]["categories"] for s in c["sections"]]
    return {
        "game_rules": game_rules["execution"] + [f"{f['function']} : {f['effect']}" for f in level_functions]
                      + [f"Nombre maximal de lignes : {game_rules['line_limits']['by_level'][str(level_id)] or 'aucune limite'}"],
        "level": level_sheet(level_id),
        "memo_sections_that_exist": memo_titles,
        "feedback_markup": FEEDBACK_MARKUP_HELP,
        "examples": [{"context": ex["context"], "expert_answer": ex["answer"]}
                     for ex in sorted(examples, key=lambda e: len(e["context"]))[:JEV_N_EXAMPLES]],
        "student_activity_and_feedback": render_feedback_context(trace_id),
    }


def jev_questions() -> dict:
    from typesafe_sdk import Choice, Noul, NoulCriteria
    questions = {}
    for q in QUESTIONS:
        w = q["widget"]
        if w["type"] == "RADIO":
            questions[q["id"]] = Choice(
                instructions={"question": q["label"]["en"], "guidance": w["label"]["en"],
                              "about": "the student in `student_activity_and_feedback`, at the moment of the help request"},
                criteria={o["value"]: f"{o['label']['en']}: {o['description']['en']}" for o in w["options"]},
            )
            continue
        for o in w["options"]:
            if o.get("exclusive"):
                continue
            questions[f"{q['id']}__{o['value']}"] = Noul(
                instructions={"question": q["label"]["en"], "guidance": w["label"]["en"],
                              "statement": f"The feedback in `student_activity_and_feedback` belongs to this category: {o['label']['en']}."},
                criteria=NoulCriteria(true=o["description"]["en"], false="This category does not apply to the feedback."),
            )
    return questions


def jev_answer(raw: dict) -> dict:
    """Turns Jev probabilities into a form answer (threshold, exclusive option when nothing applies)."""
    answers = raw["answers"]
    out = {"rationale": ""}
    for q in QUESTIONS:
        w = q["widget"]
        if w["type"] == "RADIO":
            out[q["id"]] = answers[q["id"]]["choice"]
            continue
        probs = {o["value"]: answers[f"{q['id']}__{o['value']}"]["noul"] for o in w["options"] if not o.get("exclusive")}
        ticked = [v for v, p in probs.items() if p >= JEV_THRESHOLD]
        exclusive = [o["value"] for o in w["options"] if o.get("exclusive")]
        if not ticked:
            ticked = exclusive[:1] or [max(probs, key=probs.get)]
        out[q["id"]] = ticked
        if f"{q['id']}_other" in OUTPUT_SCHEMA["properties"]:
            out[f"{q['id']}_other"] = ""
    return out


def approx_tokens(text: str) -> int:
    return int(len(text) / 3.5)


jev_sizes = [approx_tokens(json.dumps(jev_state(t), ensure_ascii=False)) for t in sample_ids]
if max(jev_sizes) > JEV_MAX_STATE_TOKENS:
    print(f"WARNING: Jev state above its limit, lower JEV_N_EXAMPLES")
print(f"Jev state size on sample contexts: {min(jev_sizes):,} to {max(jev_sizes):,} tokens (limit {JEV_MAX_STATE_TOKENS:,})")

## 7. Cache, costs and runs

Every raw response is appended to `campaign/4_llm/raw/<model>.jsonl` with the key
`(model, PROMPT_VERSION, trace_id)`. A rerun skips the keys already present, so it makes no new
call. The cost estimate below uses a rough character count (about 3.5 characters per token).
For the production run, the batch functions (Claude and GPT, half price) submit all the missing
units at once and collect the results later into the same cache.

In [ ]:
RAW_DIR.mkdir(parents=True, exist_ok=True)


def cache_path(model: str) -> Path:
    return RAW_DIR / f"{model}.jsonl"


def load_cache(model: str) -> dict[str, dict]:
    path = cache_path(model)
    if not path.exists():
        return {}
    records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    return {r["trace_id"]: r for r in records if r["prompt_version"] == PROMPT_VERSION}


def append_cache(model: str, trace_id: str, response: dict, extra: dict | None = None) -> None:
    record = {"model": model, "prompt_version": PROMPT_VERSION, "trace_id": trace_id,
              "prompt_sha256": hashlib.sha256(SYSTEM_PROMPT.encode()).hexdigest()[:16],
              "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
              "response": response, **(extra or {})}
    with open(cache_path(model), "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def call_model(model: str, trace_id: str) -> dict:
    provider = MODELS[model]
    client = client_for(provider)
    if provider == "anthropic":
        return client.messages.create(**anthropic_params(model, trace_id)).to_dict()
    if provider == "openai":
        return client.responses.create(**openai_params(model, trace_id)).to_dict()
    response = client.system_one(state=jev_state(trace_id), questions=jev_questions(), model=model)
    return response.model_dump(mode="json")


def annotate(model: str, trace_ids: list[str]) -> Counter:
    done = load_cache(model)
    status = Counter()
    for tid in trace_ids:
        if tid in done:
            status["cached"] += 1
            continue
        if DRY_RUN:
            status["skipped (DRY_RUN)"] += 1
            continue
        try:
            append_cache(model, tid, call_model(model, tid))
            status["called"] += 1
        except Exception as error:  # kept per unit: one failure must not stop the run
            status[f"error: {type(error).__name__}"] += 1
            print(f"[{model}] {tid}: {error}")
    return status

In [ ]:
PRICES_PER_MTOK = {   # input, cached input, output; None when unknown
    "claude-opus-5-5": (4.00, 0.20, 20.00),
    "gpt-6.1-sol": (2.00, 0.10, 10.00),
    "jev-1.13.0": (0.042, 0.042, 0.0),
}
ESTIMATED_OUTPUT_TOKENS = 1500   # rationale, JSON and thinking; to be replaced by measured usage

static_tokens = approx_tokens(SYSTEM_PROMPT) + sum(approx_tokens(e["context"]) + 300 for e in examples)
context_tokens = units.trace_id.head(200).map(lambda t: approx_tokens(render_feedback_context(t)))
rows = []
for split in ["dev", "test", "production"]:
    n = int((units.split == split).sum())
    for model in MODELS:
        price = PRICES_PER_MTOK.get(model)
        cost = None
        if price and n:
            if MODELS[model] == "typesafe":
                cost = n * (approx_tokens(json.dumps(jev_state(units.trace_id.iat[0]), ensure_ascii=False))) * price[0] / 1e6
            else:
                cost = (static_tokens * price[0] + (n - 1) * static_tokens * price[1]
                        + n * context_tokens.mean() * price[0] + n * ESTIMATED_OUTPUT_TOKENS * price[2]) / 1e6
        rows.append({"split": split, "model": model, "feedback": n,
                     "estimated_cost_usd": None if cost is None else round(cost, 2)})
print(f"Static part: ~{static_tokens:,} tokens; per-feedback context: ~{context_tokens.mean():,.0f} tokens on average "
      f"(max ~{context_tokens.max():,})")
pd.DataFrame(rows)

In [ ]:
if not DRY_RUN:
    from openai import OpenAI
    available = {m.id for m in OpenAI().models.list()}
    for model, provider in MODELS.items():
        if provider == "openai" and model not in available:
            raise ValueError(f"{model} is not available with this OpenAI key")

run_ids = units.loc[units.split.isin(RUN_SPLITS), "trace_id"].tolist()
for model in MODELS:
    print(model, dict(annotate(model, run_ids)))

In [ ]:
def anthropic_batch_submit(model: str, trace_ids: list[str]) -> str | None:
    """Submits the missing units as one Message Batch (half price); returns the batch id."""
    from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
    from anthropic.types.messages.batch_create_params import Request
    todo = [t for t in trace_ids if t not in load_cache(model)]
    if DRY_RUN or not todo:
        print(f"[{model}] batch not submitted ({'DRY_RUN' if DRY_RUN else 'nothing to do'}, {len(todo)} units)")
        return None
    batch = client_for("anthropic").messages.batches.create(requests=[
        Request(custom_id=t, params=MessageCreateParamsNonStreaming(**anthropic_params(model, t))) for t in todo])
    print(f"[{model}] batch {batch.id} submitted with {len(todo)} units")
    return batch.id


def anthropic_batch_collect(model: str, batch_id: str) -> Counter:
    client = client_for("anthropic")
    status = Counter()
    if client.messages.batches.retrieve(batch_id).processing_status != "ended":
        status["not ended"] += 1
        return status
    for result in client.messages.batches.results(batch_id):
        if result.result.type == "succeeded":
            append_cache(model, result.custom_id, result.result.message.to_dict(), {"batch_id": batch_id})
        status[result.result.type] += 1
    return status


def openai_batch_submit(model: str, trace_ids: list[str]) -> str | None:
    """Submits the missing units to the OpenAI Batch API (Responses endpoint); returns the batch id."""
    todo = [t for t in trace_ids if t not in load_cache(model)]
    if DRY_RUN or not todo:
        print(f"[{model}] batch not submitted ({'DRY_RUN' if DRY_RUN else 'nothing to do'}, {len(todo)} units)")
        return None
    lines = [json.dumps({"custom_id": t, "method": "POST", "url": "/v1/responses", "body": openai_params(model, t)},
                        ensure_ascii=False) for t in todo]
    batch_file = RAW_DIR / f"{model}_batch_input.jsonl"
    batch_file.write_text("\n".join(lines), encoding="utf-8")
    client = client_for("openai")
    uploaded = client.files.create(file=open(batch_file, "rb"), purpose="batch")
    batch = client.batches.create(input_file_id=uploaded.id, endpoint="/v1/responses", completion_window="24h")
    batch_file.unlink()
    print(f"[{model}] batch {batch.id} submitted with {len(todo)} units")
    return batch.id


def openai_batch_collect(model: str, batch_id: str) -> Counter:
    client = client_for("openai")
    batch = client.batches.retrieve(batch_id)
    status = Counter()
    if batch.status != "completed" or not batch.output_file_id:
        status[batch.status] += 1
        return status
    for line in client.files.content(batch.output_file_id).text.splitlines():
        item = json.loads(line)
        if item.get("response", {}).get("status_code") == 200:
            append_cache(model, item["custom_id"], item["response"]["body"], {"batch_id": batch_id})
            status["succeeded"] += 1
        else:
            status["errored"] += 1
    return status

## 8. Output

Cached responses are parsed, validated against the form and written in the FEP export format
(`expert_id` = model id). Problems found during validation are listed, not silently fixed.

In [ ]:
PARSERS = {"anthropic": anthropic_parse, "openai": openai_parse, "typesafe": jev_answer}

output_rows, issue_rows = [], []
game_of = dict(zip(units.trace_id, units.game_id))
for model, provider in MODELS.items():
    for tid, record in load_cache(model).items():
        try:
            answer = PARSERS[provider](record["response"])
        except (KeyError, ValueError, TypeError) as error:
            answer, problems = None, [f"unparsable response: {error}"]
        if answer is None:
            issue_rows.append({"model": model, "trace_id": tid, "issue": "no usable answer (refusal, truncation or parse error)"})
            continue
        clean, problems = validate_answer(answer)
        issue_rows += [{"model": model, "trace_id": tid, "issue": p} for p in problems]
        output_rows.append({"game_id": game_of.get(tid, ""), "trace_id": tid, "expert_id": model, **clean,
                            "rationale": answer.get("rationale", "")})

llm_result = pd.DataFrame(output_rows, columns=FEP_COLUMNS + ["rationale"])
issues = pd.DataFrame(issue_rows, columns=["model", "trace_id", "issue"])
if len(llm_result):
    llm_result[FEP_COLUMNS].to_csv(OUTPUT_PATH, index=False, encoding="utf-8", lineterminator="\n")
    print(f"Written: {OUTPUT_PATH.relative_to(PROJECT_ROOT)} ({len(llm_result)} rows)")
else:
    print("No cached response yet: nothing written.")
print(f"Validation issues: {len(issues)}")
issues.groupby(["model", "issue"]).size() if len(issues) else None

## 9. Evaluation

Each model is treated as one more annotator. While the prompt is tuned, the evaluation runs on the
**dev set** (`EVAL_SPLIT = "dev"`): each prompt version is scored, its errors read through the
models' rationales, the prompt or the settings adjusted, and `PROMPT_VERSION` bumped. The dev
scores of every version are kept in `campaign/4_llm/dev_scores.csv`. Once the prompt is frozen, the
**test set** is annotated and evaluated once (`RUN_SPLITS = ["test"]`, `EVAL_SPLIT = "test"`); these
are the reported scores.

For each question, Krippendorff's alpha is computed for the expert pairs, then for the pairs
(model, expert): if the model agrees with the experts as much as they agree with each other, it
codes like an expert. The alpha functions are those of notebook 03 (nominal distance for the stage,
MASI distance for the multiple-choice questions). Per-category precision, recall and F1 use the gold
labels of notebook 06.

In [ ]:
def norm_set(value):
    if not value:
        return frozenset()
    return frozenset(v.strip() for v in value.split(","))


def masi_distance(a, b):
    if not a and not b:
        return 0.0
    if a == b:
        m = 1.0
    elif a <= b or b <= a:
        m = 2 / 3
    elif a & b:
        m = 1 / 3
    else:
        m = 0.0
    jaccard = len(a & b) / len(a | b) if (a | b) else 0.0
    return 1 - jaccard * m


def nominal_distance(a, b):
    return 0.0 if a == b else 1.0


def krippendorff_alpha(pairs, distance_fn):
    n = len(pairs)
    observed = sum(distance_fn(v1, v2) for v1, v2 in pairs) / n
    pooled = [v for pair in pairs for v in pair]
    total = len(pooled)
    expected_sum = sum(
        distance_fn(pooled[i], pooled[j]) for i in range(total) for j in range(total) if i != j
    )
    expected = expected_sum / (total * (total - 1))
    return float("nan") if expected == 0 else 1 - observed / expected


def cohen_kappa(values_a, values_b):
    """Wrapper kept so per-category tables read the same whichever question they come from."""
    if values_a.nunique() == 1 and values_b.nunique() == 1 and values_a.iloc[0] == values_b.iloc[0]:
        return 1.0
    return cohen_kappa_score(values_a, values_b)

In [ ]:
if expert_result is None or gold is None or llm_result.empty:
    print("Evaluation skipped: gold standard or model annotations not available yet.")
else:
    eval_ids = set(units.loc[units.split == EVAL_SPLIT, "trace_id"])
    experts_eval = expert_result[expert_result.trace_id.isin(eval_ids)]
    rows = []
    for q in QUESTIONS:
        col = q["id"]
        single = q["widget"]["type"] == "RADIO"
        to_value = (lambda v: v) if single else norm_set
        distance = nominal_distance if single else masi_distance
        by_trace = experts_eval.groupby("trace_id")[col].apply(list)
        expert_pairs = [(to_value(v[0]), to_value(v[1])) for v in by_trace if len(v) == 2]
        rows.append({"question": col, "annotator": "experts (pairs)", "n_pairs": len(expert_pairs),
                     "alpha": round(krippendorff_alpha(expert_pairs, distance), 3)})
        for model in MODELS:
            mod = llm_result[(llm_result.expert_id == model) & llm_result.trace_id.isin(eval_ids)].set_index("trace_id")[col]
            pairs = [(to_value(mod[t]), to_value(v)) for t, values in by_trace.items() if t in mod.index for v in values]
            if pairs:
                rows.append({"question": col, "annotator": f"{model} vs expert", "n_pairs": len(pairs),
                             "alpha": round(krippendorff_alpha(pairs, distance), 3)})
    agreement = pd.DataFrame(rows)
    print(f"Evaluation subset: {EVAL_SPLIT} ({len(eval_ids)} feedback), prompt {PROMPT_VERSION}")
    if EVAL_SPLIT == "dev":
        scores = agreement.assign(prompt_version=PROMPT_VERSION,
                                  evaluated_at=datetime.now(timezone.utc).isoformat(timespec="seconds"))
        if DEV_SCORES_PATH.exists():
            previous = pd.read_csv(DEV_SCORES_PATH, dtype={"prompt_version": str})
            scores = pd.concat([previous[previous.prompt_version != PROMPT_VERSION], scores], ignore_index=True)
        scores.to_csv(DEV_SCORES_PATH, index=False, encoding="utf-8", lineterminator="\n")
        print(f"Dev scores of {PROMPT_VERSION} written to {DEV_SCORES_PATH.relative_to(PROJECT_ROOT)}")
    display(agreement.pivot(index="annotator", columns="question", values="alpha"))

    reference = gold[gold.trace_id.isin(eval_ids)].set_index("trace_id")
    per_category = []
    for model in MODELS:
        mod = llm_result[(llm_result.expert_id == model) & llm_result.trace_id.isin(reference.index)].set_index("trace_id")
        for q in QUESTIONS:
            for o in q["widget"]["options"]:
                truth = reference.loc[mod.index, q["id"]].map(lambda v: o["value"] in norm_set(v)).astype(bool)
                pred = mod[q["id"]].map(lambda v: o["value"] in norm_set(v)).astype(bool)
                tp, fp, fn = int((truth & pred).sum()), int((~truth & pred).sum()), int((truth & ~pred).sum())
                precision = tp / (tp + fp) if tp + fp else float("nan")
                recall = tp / (tp + fn) if tp + fn else float("nan")
                f1 = 2 * precision * recall / (precision + recall) if precision + recall else float("nan")
                per_category.append({"model": model, "category": o["value"], "support": int(truth.sum()),
                                     "precision": round(precision, 2), "recall": round(recall, 2), "f1": round(f1, 2)})
    display(pd.DataFrame(per_category).pivot(index="category", columns="model", values="f1"))

## 10. Summary

In [ ]:
summary = [
    {"step": "Annotation units (B/C feedback)", "value": len(units)},
    {"step": "  - example pool / dev / test (gold games)", "value": " / ".join(str(int((units.split == s).sum())) for s in ["examples", "dev", "test"])},
    {"step": "  - production (outside gold)", "value": int((units.split == "production").sum())},
    {"step": "Few-shot examples", "value": len(examples)},
    {"step": f"Static prompt {PROMPT_VERSION} (approx. tokens)", "value": approx_tokens(SYSTEM_PROMPT)},
    {"step": "Mode", "value": "DRY_RUN (no API call)" if DRY_RUN else "live"},
]
for model in MODELS:
    summary.append({"step": f"Cached responses, {model}", "value": len(load_cache(model))})
summary.append({"step": "Rows written to llm_result.csv", "value": len(llm_result)})
summary.append({"step": "Validation issues", "value": len(issues)})
pd.DataFrame(summary)